# Ray recovery experiments on GCP

**Project:** `research-488322` · **Image:** `ray-built-v1` · **Zone:** `us-central1-a`

This notebook creates **five Standard e2-standard-4 VMs (4 vCPUs each)**, runs matched CPU recovery experiments, downloads reports and PNG/PDF plots, and includes optional deletion at the end. It uses the Ray build already stored in your image; it does not rebuild Ray or change that image.

**Run cells in order.** Run this notebook outside the disposable experiment VMs. Five VMs remain billable until you execute cleanup, even if the notebook disconnects or a test fails.

Two comparisons are included:

| Comparison | What it measures |
|---|---|
| Ordinary Ray DDP checkpoint retry vs TorchFT + aligned Ray Data | Combined integration benefit and healthy overhead |
| TorchFT + indexed input vs TorchFT + retained aligned input | Input strategy contribution with the same peer recovery |

This suite uses synthetic CPU training. It does **not** enable Fixed-R. The first comparison includes TorchFT's existing recovery capabilities. Negative results and failed correctness checks remain visible.

**Failure scope:** SIGKILL of the entire Ray worker service on one VM, including its actors and object store. The VM itself stays running. The worker service stays down for the rest of that observation. Head and shared checkpoint storage survive.

**Validation status:** source reviewed, Python compilation and offline orchestration tests passed; no live GCP execution was performed while preparing this notebook. Start with the smoke tests.


## 1. Authenticate the notebook host

Requirements: Python 3.10+, Jupyter/IPython, [Google Cloud CLI](https://cloud.google.com/sdk/docs/install), and OpenSSH (`ssh`, `ssh-keygen`). Ray/PyTorch are needed on the saved image, not on this notebook host.

In a terminal **on the notebook host**, authenticate once if needed:

```bash
gcloud auth login --no-launch-browser
gcloud config set project research-488322
```

The account needs Compute resource management, VM metadata/SSH access, and **IAP-secured Tunnel User** access. This notebook uses an isolated VPC and IAP SSH; it never opens Ray ports to the public internet. The VMs use external IPs for package downloads. Metadata-based SSH is requested; an organization enforcing OS Login needs an adapted setup.

The next cell checks the active login and enables the Compute/IAP APIs. It creates no VMs.


In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

for tool in ("gcloud", "ssh", "ssh-keygen"):
    if shutil.which(tool) is None:
        raise RuntimeError(f"Install {tool} on the notebook host and restart Jupyter if PATH changed.")

def cli(*args):
    p = subprocess.run(["gcloud", *args], text=True, capture_output=True)
    if p.returncode:
        raise RuntimeError(p.stderr)
    return p.stdout

print(cli("auth", "list", "--filter=status:ACTIVE", "--format=table(account,status)"))
print(cli("services", "enable", "compute.googleapis.com", "iap.googleapis.com",
          "--project=research-488322", "--quiet"))


## 2. Visible Python source

Every helper script is printed in full in the following code cells. Nothing is encoded or hidden. Each `%%writefile` cell writes its visible contents to a local `.py` file; it does not run that script or create cloud resources. Run these cells in order before importing the controller.

You can edit the code directly in these cells. Re-running a `%%writefile` cell overwrites the corresponding helper file. To apply controller edits after it has been imported, restart the kernel and rerun the source cells, then reconnect using your saved `STATE_FILE`.


In [ ]:
from pathlib import Path
import sys

BUNDLE_DIR = Path("ray_gcp_harness_visible").resolve()
(BUNDLE_DIR / "_support").mkdir(parents=True, exist_ok=True)
print("Visible helper scripts will be written to:", BUNDLE_DIR)


### Worker service control

`node_control.py` — Starts, stops and abruptly kills only the experiment Ray service.


In [ ]:
%%writefile ray_gcp_harness_visible/node_control.py
"""Root-owned forced SSH command; controls only the experiment Ray service."""
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import subprocess
import sys
import time

CONFIG = Path('/etc/ray-gcp.json')
UNIT = 'ray-exp-node.service'
ENV = '/opt/ray-env/ray-dev/bin'
REPO = Path('/opt/ray-work/ray')
HASH_PATHS = [
    'python/ray/train/v2/_internal/data_integration/step_aligned_input.py',
    'python/ray/train/v2/_internal/execution/worker_group/worker_group.py',
    'python/ray/train/v2/_internal/callbacks/datasets.py',
    'python/ray/train/v2/examples/pytorch/torchft_linear_example.py',
    'python/ray/train/v2/torch/torchft_config.py',
]


def run(command, check=True):
    return subprocess.run(command, check=check, text=True, capture_output=True, timeout=60)


def active():
    return run(['systemctl', 'is-active', UNIT], False).stdout.strip()


def stop():
    status = active()
    if status not in ('inactive', 'unknown', 'failed', ''):
        run(['systemctl', 'stop', UNIT])
    if active() in ('active', 'activating', 'deactivating'):
        raise RuntimeError('Ray service did not stop')
    run(['systemctl', 'reset-failed', UNIT], False)


def main():
    config = json.loads(CONFIG.read_text())
    request = json.load(sys.stdin)
    if request.get('run_id') != config['run_id']:
        raise ValueError('Experiment identity mismatch')
    action = request['action']
    if action == 'identity':
        import ray
        import ray._raylet
        import torch
        from torchft import Manager, Optimizer, ProcessGroupGloo
        from torchft.checkpointing.pg_transport import PGTransport
        from ray.train.v2._internal.data_integration.step_aligned_input import StepAlignedDataConfig
        if not Path(ray.__file__).resolve().is_relative_to(REPO / 'python'):
            raise RuntimeError('Ray is not imported from the compiled checkout')
        result = dict(name=config['name'], ip=config['ip'], python=sys.executable,
                      ray_file=ray.__file__, cpu_count=os.cpu_count(),
                      versions={p: importlib.metadata.version(p) for p in
                                ['ray', 'torch', 'torchft', 'numpy', 'pyarrow']},
                      git_commit=run(['git', '-c', 'safe.directory='+str(REPO),
                                      '-C', str(REPO), 'rev-parse', 'HEAD']).stdout.strip(),
                      source_sha256={p: hashlib.sha256((REPO / p).read_bytes()).hexdigest()
                                     for p in HASH_PATHS},
                      native_sha256=hashlib.sha256(Path(ray._raylet.__file__).read_bytes()).hexdigest())
    elif action == 'start':
        stop()
        cmd = ['systemd-run', '--quiet', '--collect', '--unit='+UNIT,
               '--uid=raybench', '--gid=raybench', '--property=KillMode=control-group',
               '--property=TimeoutStopSec=10', '--property=Restart=no',
               '--setenv=PATH='+ENV+':/usr/bin:/bin',
               '--setenv=RAY_TRAIN_V2_ENABLED=1', '--setenv=RAY_USAGE_STATS_ENABLED=0',
               '--setenv=OMP_NUM_THREADS=1', '--setenv=MKL_NUM_THREADS=1',
               '--setenv=OPENBLAS_NUM_THREADS=1',
               ENV+'/ray', 'start', '--block', '--disable-usage-stats',
               '--node-ip-address='+config['ip'], '--object-store-memory=1073741824',
               '--num-cpus='+('2' if config['head'] else '4')]
        if config['head']:
            cmd += ['--head', '--port=6379', '--include-dashboard=false',
                    '--temp-dir=/tmp/ray-gcp']
        else:
            cmd += ['--address='+config['head_ip']+':6379', '--resources={"train_slot":1}']
        run(cmd)
        result = {'started': True, 'name': config['name']}
    elif action == 'kill':
        if config['head']:
            raise ValueError('Head failure is outside this experiment')
        if active() != 'active':
            raise RuntimeError('Cannot inject failure into an inactive service')
        run(['systemctl', 'kill', '--kill-whom=all', '--signal=SIGKILL', UNIT])
        deadline = time.monotonic() + 15
        while active() in ('active', 'activating', 'deactivating'):
            if time.monotonic() > deadline:
                raise RuntimeError('Worker service remained active after SIGKILL')
            time.sleep(.1)
        result = {'killed': True, 'name': config['name'], 'active': active()}
    elif action == 'stop':
        stop()
        result = {'stopped': True}
    elif action == 'logs':
        result = {'journal': run(['journalctl', '-u', UNIT, '--no-pager', '-n', '2000']).stdout}
    else:
        raise ValueError('Unsupported action')
    print(json.dumps(result), flush=True)


if __name__ == '__main__':
    main()


### Support package

`_support/__init__.py` — Defines the local support package.


In [ ]:
%%writefile ray_gcp_harness_visible/_support/__init__.py
"""GCP benchmark support modules."""


### Physical GCP cluster backend

`_support/gcp_cluster.py` — Connects the head and worker services, checks identities, injects the worker failure and collects service logs.


In [ ]:
%%writefile ray_gcp_harness_visible/_support/gcp_cluster.py
"""Physical GCP transport for the matched upstream benchmark.
The driver and recorder share the surviving head clock. No heartbeat tuning.
"""
from concurrent.futures import ThreadPoolExecutor
import json
from pathlib import Path
import socket
import subprocess
import time
from types import SimpleNamespace

CONFIG = Path('/etc/ray-gcp.json')


def command(node, action):
    c = json.loads(CONFIG.read_text())
    payload = json.dumps({'run_id': c['run_id'], 'action': action})
    if node['head']:
        cmd = ['sudo', '-n', '/opt/ray-env/ray-dev/bin/python',
               '/opt/ray-gcp/node_control.py']
    else:
        cmd = ['ssh', '-i', '/var/lib/raybench/.ssh/id_ed25519',
               '-o', 'BatchMode=yes', '-o', 'ConnectTimeout=10',
               '-o', 'StrictHostKeyChecking=yes',
               '-o', 'UserKnownHostsFile=/var/lib/raybench/.ssh/known_hosts',
               'root@'+node['ip']]
    p = subprocess.run(cmd, input=payload, capture_output=True, text=True, timeout=120)
    if p.returncode:
        raise RuntimeError(f"{node['name']} {action}: {p.stderr[-3000:]}")
    return json.loads(p.stdout)


class GCPCluster:
    def __init__(self, folder):
        self.folder = Path(folder)
        self.config = json.loads(CONFIG.read_text())
        self.hosts = self.config['nodes']
        self.nodes = []
        self.metadata = {}

    def start(self):
        import ray
        # Also cleans up services left by an interrupted previous observation.
        self.shutdown()
        with ThreadPoolExecutor(max_workers=5) as pool:
            identities = list(pool.map(lambda n: command(n, 'identity'), self.hosts))
        signatures = [{k: v for k, v in i.items() if k not in ('name', 'ip')}
                      for i in identities]
        if any(s != signatures[0] for s in signatures[1:]):
            raise RuntimeError('VM source, native build or package versions differ')
        if any(i['cpu_count'] != 4 for i in identities):
            raise RuntimeError('This experiment requires four-vCPU VMs')
        head = next(n for n in self.hosts if n['head'])
        command(head, 'start')
        deadline = time.monotonic() + 90
        while True:
            try:
                with socket.create_connection((head['ip'], 6379), timeout=1):
                    break
            except OSError:
                if time.monotonic() > deadline:
                    raise TimeoutError('Ray head startup timed out')
                time.sleep(.5)
        self.address = head['ip']+':6379'
        ray.init(address=self.address)
        workers = [n for n in self.hosts if not n['head']]
        with ThreadPoolExecutor(max_workers=4) as pool:
            list(pool.map(lambda n: command(n, 'start'), workers))
        deadline = time.monotonic() + 120
        while True:
            alive = [n for n in ray.nodes() if n['Alive']]
            if len(alive) == len(self.hosts):
                break
            if time.monotonic() > deadline:
                raise TimeoutError('Not all physical workers registered')
            time.sleep(.5)
        for worker in workers:
            matches = [n for n in alive if n['NodeManagerAddress'] == worker['ip']]
            if len(matches) != 1 or matches[0]['Resources'].get('train_slot') != 1:
                raise RuntimeError('Missing or ambiguous worker registration')
            self.nodes.append(SimpleNamespace(node_id=matches[0]['NodeID'], **worker))
        self.metadata = dict(hosts=self.hosts, installations=identities,
            node_mapping=[vars(n) for n in self.nodes], cpus_per_node=4,
            training_workers=2, training_cpus_per_worker=3, spare_worker_hosts=2,
            address=self.address, run_id=self.config['run_id'],
            fault_model='SIGKILL entire Ray worker systemd cgroup; VM stays up',
            clock='recorder arrival and driver on the same surviving head',
            storage='NFSv4 on surviving head; identical checkpoint storage in both arms')
        return self.nodes

    def remove_node(self, node, allow_graceful=False):
        if allow_graceful or node not in self.nodes or node.head:
            raise ValueError('Only abrupt loss of a known worker is permitted')
        self.metadata['kill_dispatch_time'] = time.monotonic()
        result = command(vars(node), 'kill')
        self.metadata['kill_ack_time'] = time.monotonic()
        self.metadata['kill_result'] = result
        self.metadata['killed_node'] = vars(node)
        # This worker is not restarted until the next independent observation.

    def shutdown(self):
        errors = []
        for node in self.hosts:
            try:
                log = command(node, 'logs')
                directory = self.folder / 'nodes'
                directory.mkdir(parents=True, exist_ok=True)
                (directory / (node['name']+'.log')).write_text(log['journal'])
                command(node, 'stop')
            except Exception as exc:
                errors.append(str(exc))
        if errors:
            raise RuntimeError('Cluster cleanup failed: '+'; '.join(errors))


### Training experiment and correctness checks

`_support/ordinary_ray_comparison.py` — Full matched MLP training loop, checkpoint handling, TorchFT integration, recovery metrics and validation.


In [ ]:
%%writefile ray_gcp_harness_visible/_support/ordinary_ray_comparison.py
"""Matched CPU MLP comparison: ordinary Ray DDP retry versus TorchFT peer retry.

Synthetic classification isolates training-system cost, not predictive quality.
Both modes write the same scheduled model/Adam/step checkpoints. Input for the
ordinary mode is directly indexed by restored step (no prefix replay penalty).
"""
import concurrent.futures
import hashlib
import importlib.metadata
import json
from pathlib import Path
import tempfile
import time


def run(args, write_json):
    import numpy as np
    import ray
    import ray.data
    import ray.train as train
    import torch
    from torch import nn
    from datetime import timedelta
    from ray.cluster_utils import Cluster
    from ray.train import Checkpoint, RunConfig, ScalingConfig, UserCallback
    from ray.train.torch import TorchConfig, TorchTrainer
    from ray.train.v2.api.config import FailureConfig
    from ray.train.v2._internal.data_integration.step_aligned_input import StepAlignedDataConfig
    from ray.train.v2.torch.torchft_config import TorchftConfig
    from ray.train.v2.examples.pytorch.torchft_linear_example import _FailureAwareDistributedDataParallel
    from ray.util.scheduling_strategies import NodeAffinitySchedulingStrategy

    aligned = args.mode == "aligned"
    peer_recovery = args.mode != "ordinary"
    folder = Path(args.sample).parent
    text_manifest = None
    if args.text_manifest:
        from _support import text_stream
        # Ray workers do not need this benchmark directory installed as a package.
        ray.cloudpickle.register_pickle_by_value(text_stream)
        text_manifest, text_root = text_stream.load_manifest(args.text_manifest, args.steps, args.batch_size)
    if args.profile_recovery:
        from _support import recovery_trace
        ray.cloudpickle.register_pickle_by_value(recovery_trace)
    num_classes = text_manifest["num_classes"] if text_manifest else 10

    @ray.remote(num_cpus=0)
    class Recorder:
        def __init__(self):
            self.events = []
            self.node = None
            self.finals = {}
            self.checkpoints = {}

        def add(self, event):
            event = dict(event)
            if getattr(args, "gcp", False):
                event["time"] = time.monotonic()
            else:
                event.setdefault("time", time.monotonic())
            self.events.append(event)

        def acknowledge(self, step, path):
            self.checkpoints[step] = path
            self.events.append({"kind": "checkpoint_ack", "step": step,
                                "path": path, "time": time.monotonic()})

        def acknowledged(self, step):
            return step in self.checkpoints

        def claim(self, node):
            if self.node is not None:
                return False
            self.node = node
            return True

        def snapshot(self):
            return self.node, self.events, self.finals

        def finish(self, rank, result):
            self.finals[rank] = result

    def batch_values(ids):
        # Stateless deterministic source: cheap random access for ordinary Ray.
        # Both modes receive the exact same features and labels for each step.
        ids = np.asarray(ids, dtype=np.int64)
        if text_manifest is not None:
            pieces = []
            for chunk in dict.fromkeys((ids // args.batch_size).tolist()):
                batch, event = text_stream.read_batch(text_root, text_manifest, chunk)
                # Same accounting RPC per decoded chunk in both strategies.
                ray.get(recorder.add.remote(event))
                pieces.append(batch)
            complete = {key: np.concatenate([b[key] for b in pieces]) for key in ("id", "x", "y")}
            positions = {int(value): index for index, value in enumerate(complete["id"])}
            indices = [positions[int(value)] for value in ids]
            return {key: value[indices] for key, value in complete.items()}
        features = np.sin(ids[:, None] * .013 + np.arange(256)[None, :] * .071).astype(np.float32)
        labels = ((ids // 7 + ids % 3) % 10).astype(np.int64)
        return {"x": features, "y": labels, "id": ids}

    physical = getattr(args, "gcp", False)
    if physical:
        from _support.gcp_cluster import GCPCluster
        cluster = GCPCluster(folder)
    else:
        cluster = Cluster()
    pool = concurrent.futures.ThreadPoolExecutor(max_workers=1)
    try:
        if physical:
            nodes = cluster.start()
            write_json(folder / "cluster.json", cluster.metadata)
        else:
            cluster.add_node(num_cpus=2, include_dashboard=False)
            nodes = [cluster.add_node(num_cpus=1, resources={"train_slot": 1}) for _ in range(3)]
            ray.init(address=cluster.address)
        recorder = Recorder.options(scheduling_strategy=NodeAffinitySchedulingStrategy(
            ray.get_runtime_context().get_node_id(), soft=False)).remote()

        class CheckpointAcknowledgement(UserCallback):
            def after_exception(self, run_context, worker_exceptions):
                if args.profile_recovery:
                    ray.get(recorder.add.remote({"kind": "controller_observed_failure",
                        "time": time.monotonic(), "ranks": sorted(worker_exceptions),
                        "errors": {rank: type(error).__name__ for rank, error in worker_exceptions.items()}}))

            def after_report(self, run_context, metrics, checkpoint):
                if args.profile_recovery:
                    recovery_trace.install(recorder)
                if checkpoint is None:
                    return
                steps = {m["step"] for m in metrics}
                if len(steps) != 1:
                    raise RuntimeError(f"Checkpoint report steps disagree: {steps}")
                step = steps.pop()
                with checkpoint.as_directory() as directory:
                    state = torch.load(Path(directory) / "state.pt",
                                       map_location="cpu", weights_only=True)
                if state["step"] != step:
                    raise RuntimeError(f"Persisted checkpoint is not step {step}")
                # CheckpointManager runs before this callback. A worker-side
                # report return alone does not prove controller registration.
                ray.get(recorder.acknowledge.remote(step, str(checkpoint.path)))

        def training(config):
            rank = train.get_context().get_world_rank()
            # Deliberate simulated initialization, identical per invocation in
            # both modes. Concurrent worker startup delays are not summed.
            ray.get(recorder.add.remote({"kind": "start", "rank": rank,
                "node_id": ray.get_runtime_context().get_node_id()}))
            time.sleep(args.startup_delay_s)
            torch.set_num_threads(3 if physical else 1)
            torch.manual_seed(100 + args.repeat)
            torch.use_deterministic_algorithms(True)
            base = nn.Sequential(nn.Linear(256, args.model_width), nn.ReLU(),
                nn.Linear(args.model_width, args.model_width), nn.ReLU(),
                nn.Linear(args.model_width, args.model_width), nn.ReLU(),
                nn.Linear(args.model_width, num_classes))
            raw_optimizer = torch.optim.Adam(base.parameters(), lr=1e-3)
            checkpoint = train.get_checkpoint()
            restored_step = 0
            if checkpoint:
                with checkpoint.as_directory() as path:
                    state = torch.load(Path(path) / "state.pt", map_location="cpu", weights_only=True)
                    base.load_state_dict(state["model"])
                    raw_optimizer.load_state_dict(state["optimizer"])
                    restored_step = state["step"]
            ray.get(recorder.add.remote({"kind": "initialized", "rank": rank, "restored_step": restored_step}))
            manager = None
            if peer_recovery:
                from torchft import Manager, Optimizer, ProcessGroupGloo
                from torchft.checkpointing.pg_transport import PGTransport
                pg = ProcessGroupGloo(timeout=timedelta(seconds=30))

                def state_dict():
                    return {"model": base.state_dict(), "optim": raw_optimizer.state_dict()}

                def load_state_dict(state):
                    if args.profile_recovery:
                        recorder.add.remote({"kind": "peer_state_load_begin", "rank": rank, "time": time.monotonic()})
                    base.load_state_dict(state["model"])
                    raw_optimizer.load_state_dict(state["optim"])
                    if args.profile_recovery:
                        recorder.add.remote({"kind": "peer_state_load_end", "rank": rank, "time": time.monotonic()})

                if args.profile_recovery:
                    recorder.add.remote({"kind": "torchft_setup_begin", "rank": rank, "time": time.monotonic()})
                manager = Manager(pg=pg, min_replica_size=2, world_size=1, rank=0,
                    replica_id=f"compare_{rank}", use_async_quorum=False,
                    timeout=timedelta(seconds=60), state_dict=state_dict, load_state_dict=load_state_dict,
                    checkpoint_transport=PGTransport(pg, timeout=timedelta(seconds=30), device=torch.device("cpu")))
                model = _FailureAwareDistributedDataParallel(manager, base)
                optimizer = Optimizer(manager, raw_optimizer)
                shard = train.get_dataset_shard("train") if aligned else None
                if args.profile_recovery:
                    recorder.add.remote({"kind": "torchft_setup_end", "rank": rank, "time": time.monotonic()})
            else:
                model = train.torch.prepare_model(base)
                optimizer = raw_optimizer
            step = restored_step
            checkpointed_step = restored_step if checkpoint else -1

            def save(step):
                started = time.monotonic()
                with tempfile.TemporaryDirectory() as temporary:
                    cp = None
                    byte_count = 0
                    if rank == 0:
                        path = Path(temporary) / "state.pt"
                        torch.save({"model": base.state_dict(), "optimizer": raw_optimizer.state_dict(),
                                    "step": step}, path)
                        byte_count = path.stat().st_size
                        cp = Checkpoint.from_directory(temporary)
                    train.report({"step": step}, checkpoint=cp)
                    deadline = time.monotonic() + 60
                    while not ray.get(recorder.acknowledged.remote(step)):
                        if time.monotonic() >= deadline:
                            raise TimeoutError(f"Controller did not acknowledge checkpoint {step}")
                        time.sleep(.01)
                    ray.get(recorder.add.remote({"kind": "checkpoint", "rank": rank,
                        "step": step, "seconds": time.monotonic() - started, "bytes": byte_count}))

            try:
                while True:
                    sync_started = time.monotonic()
                    optimizer.zero_grad()
                    if peer_recovery:
                        manager.wait_quorum()
                        step = manager.current_step()
                        if manager.num_participants() != 2:
                            raise RuntimeError("Full membership required")
                    if args.profile_recovery and time.monotonic() - sync_started >= .05:
                        recorder.add.remote({"kind": "step_synchronization", "rank": rank, "step": step,
                            "time": sync_started, "end_time": time.monotonic()})
                    if step >= args.steps:
                        break
                    # Scheduled durable checkpoints in BOTH arms. On replacement
                    # the healed peer step skips already-completed checkpoints.
                    if step % args.checkpoint_every == 0 and step > checkpointed_step:
                        save(step)
                        checkpointed_step = step
                    ids = list(range((step * 2 + rank) * args.batch_size,
                                     (step * 2 + rank + 1) * args.batch_size))
                    batch = shard.batch_for_step(step) if aligned else batch_values(ids)
                    if batch["id"].tolist() != ids:
                        raise RuntimeError("Input does not match the committed model step")
                    if text_manifest is not None:
                        ray.get(recorder.add.remote({"kind": "input_batch", "rank": rank,
                            "step": step, "digest": text_stream.digest(batch)}))
                    if args.fault and rank == args.failed_rank and step == args.fault_step:
                        if ray.get(recorder.claim.remote(ray.get_runtime_context().get_node_id())):
                            while True:
                                time.sleep(.1)
                    compute_started = time.monotonic()
                    loss = nn.functional.cross_entropy(model(torch.tensor(batch["x"])), torch.tensor(batch["y"]))
                    loss.backward()
                    optimizer.step()
                    next_step = manager.current_step() if peer_recovery else step + 1
                    # An aborted update is not a committed step and is not counted
                    # as successful recomputation. Its elapsed work remains timed.
                    if next_step > step:
                        ray.get(recorder.add.remote({"kind": "update", "rank": rank, "step": next_step,
                            "compute_s": time.monotonic() - compute_started}))
                    step = next_step
                flat = np.concatenate([p.detach().cpu().numpy().reshape(-1) for p in base.parameters()])
                model_path = folder / f"model-rank-{rank}.npy"
                np.save(model_path, flat)
                ray.get(recorder.finish.remote(rank, {"step": step, "model_path": str(model_path)}))
                train.report({"step": step, "final": True})
                ray.get(recorder.add.remote({"kind": "finish", "rank": rank}))
            finally:
                if manager is not None:
                    manager.shutdown()

        dataset_args = {}
        if aligned:
            range_options = {"override_num_blocks": args.steps} if text_manifest is not None else {}
            ds = ray.data.range(args.steps * 2 * args.batch_size, **range_options).map_batches(
                lambda b: batch_values(b["id"]), batch_format="numpy", batch_size=2 * args.batch_size)
            dataset_args = {"datasets": {"train": ds}, "dataset_config": StepAlignedDataConfig(
                args.batch_size, synchronous_full_membership=True)}
        trainer = TorchTrainer(training, scaling_config=ScalingConfig(num_workers=2,
            placement_strategy="STRICT_SPREAD", resources_per_worker={"CPU": 3 if physical else 1, "train_slot": 1}),
            torch_config=(TorchftConfig(backend="gloo", lighthouse_kwargs={"min_replicas": 2,
                "bind": ray.util.get_node_ip_address() + ":0"}) if peer_recovery
                          else TorchConfig(backend="gloo", timeout_s=60)),
            run_config=RunConfig(storage_path=str(folder / "storage"),
                failure_config=FailureConfig(max_failures=1),
                callbacks=[CheckpointAcknowledgement()]), **dataset_args)
        started = time.monotonic()
        future = pool.submit(trainer.fit)
        fault_time = None
        if args.fault:
            while not future.done():
                node, _, _ = ray.get(recorder.snapshot.remote())
                if node is not None:
                    fault_time = time.monotonic()
                    if args.profile_recovery:
                        ray.get(recorder.add.remote({"kind": "node_kill_requested", "time": fault_time, "node": node}))
                    cluster.remove_node(next(n for n in nodes if n.node_id == node), allow_graceful=False)
                    if physical:
                        write_json(folder / "cluster.json", cluster.metadata)
                    if args.profile_recovery:
                        ray.get(recorder.add.remote({"kind": "node_kill_returned", "time": time.monotonic()}))
                        # This observes the driver's node table, not the exact GCS
                        # detection timestamp. Bound observation and retain failures.
                        while not future.done() and time.monotonic() - fault_time < 60:
                            if any(n["NodeID"] == node and not n["Alive"] for n in ray.nodes()):
                                ray.get(recorder.add.remote({"kind": "driver_observed_node_dead", "time": time.monotonic()}))
                                break
                            time.sleep(.1)
                    break
                if time.monotonic() - started > args.timeout_s - 30:
                    raise TimeoutError("Fault gate was not reached")
                time.sleep(.1)
            if fault_time is None:
                future.result()
                raise RuntimeError("Requested fault was not injected")
        result = future.result(timeout=args.timeout_s)
        ended = time.monotonic()
        _, events, finals = ray.get(recorder.snapshot.remote())
        if result.error or len(finals) != 2 or any(v["step"] != args.steps for v in finals.values()):
            raise RuntimeError("Training did not finish correctly")
        starts = [e for e in events if e["kind"] == "start"]
        counts = {r: sum(e["rank"] == r for e in starts) for r in (0, 1)}
        expected = {r: 1 + int(args.fault and (not peer_recovery or r == args.failed_rank)) for r in (0, 1)}
        if counts != expected:
            raise RuntimeError(f"Unexpected restart counts: {counts}, expected {expected}")
        arrays = [np.load(finals[r]["model_path"]) for r in (0, 1)]
        if not np.allclose(arrays[0], arrays[1], atol=1e-5, rtol=0):
            raise RuntimeError("Final replicas disagree")
        updates = [e for e in events if e["kind"] == "update"]
        repeated = {r: len([e for e in updates if e["rank"] == r]) - args.steps for r in (0, 1)}
        recovery = None
        if fault_time is not None:
            # Both ranks have advanced past the last pre-failure model step.
            recovery = max(min(e["time"] for e in updates if e["rank"] == r
                               and e["step"] > args.fault_step and e["time"] >= fault_time)
                           for r in (0, 1)) - fault_time
        output = {"status": "completed", "mode": args.mode, "fault": args.fault,
            "repeat": args.repeat, "fit_s": ended - started,
            "loop_span_s": max(e["time"] for e in events if e["kind"] == "finish") - min(e["time"] for e in starts),
            "startup_delay_s": args.startup_delay_s, "profile_recovery": args.profile_recovery,
            "recovery_to_new_progress_s": recovery, "repeated_updates_per_rank": repeated,
            "fault_time": fault_time, "events": events, "final": finals, "starts": counts,
            "checkpoint_bytes": sum(e["bytes"] for e in events if e["kind"] == "checkpoint"),
            "versions": {"ray": ray.__version__, "torch": torch.__version__,
                         "torchft": importlib.metadata.version("torchft")}}
        if text_manifest is not None:
            preprocessing = [e for e in events if e["kind"] == "preprocess"]
            counts = {}
            for event in preprocessing:
                counts[event["chunk"]] = counts.get(event["chunk"], 0) + 1
            output["input_work"] = {
                "decoded_chunks": len(preprocessing),
                "unique_chunks": len(counts),
                "redecoded_chunks": sum(max(0, count - 1) for count in counts.values()),
                "compressed_bytes_read": sum(e["compressed_bytes"] for e in preprocessing),
                "preprocessing_worker_seconds": sum(e["seconds"] for e in preprocessing)}
        if physical:
            first_nodes = [next(e["node_id"] for e in starts if e["rank"] == r) for r in (0, 1)]
            allowed = {n.node_id for n in nodes}
            if len(set(first_nodes)) != 2 or not set(first_nodes) <= allowed:
                raise RuntimeError("Training ranks were not on distinct physical workers")
            if args.fault:
                killed = cluster.metadata["killed_node"]["node_id"]
                later = [e for e in starts if e["time"] >= fault_time]
                if any(e["node_id"] == killed or e["node_id"] not in allowed for e in later):
                    raise RuntimeError("Replacement did not use a surviving physical worker")
            output["cluster"] = cluster.metadata
        write_json(args.sample, output)
    finally:
        ray.shutdown()
        cluster.shutdown()
        pool.shutdown(wait=False, cancel_futures=True)


def recovery_errors(sample, parameters):
    """Audit raw evidence, including old reports that lack acknowledgement events."""
    errors = []
    if sample.get("status") != "completed":
        return ["run did not complete"]
    fault = sample["fault"]
    ordinary = sample["mode"] == "ordinary"
    fault_step = parameters["fault_step"]
    latest = (fault_step // parameters["checkpoint_every"]) * parameters["checkpoint_every"]
    events = sample.get("events", [])
    acknowledgements = [e for e in events if e["kind"] == "checkpoint_ack"]
    for step in range(0, parameters["steps"], parameters["checkpoint_every"]):
        if not any(e["step"] == step for e in acknowledgements):
            errors.append(f"no controller acknowledgement for checkpoint {step}")
    if fault and not any(e["step"] == latest and e["time"] < sample["fault_time"]
                         for e in acknowledgements):
        errors.append(f"checkpoint {latest} was not acknowledged before node loss")
    for rank in (0, 1):
        initializations = [e for e in events if e["kind"] == "initialized" and e["rank"] == rank]
        replaced = fault and (ordinary or rank == parameters["failed_rank"])
        expected_restores = [0, latest] if replaced else [0]
        restores = [e["restored_step"] for e in initializations]
        if restores != expected_restores:
            errors.append(f"rank {rank} restored {restores}, expected {expected_restores}")
        repeated = sum(e["kind"] == "update" and e.get("rank") == rank for e in events) - parameters["steps"]
        expected_repeats = fault_step - latest if fault and ordinary else 0
        if repeated != expected_repeats:
            errors.append(f"rank {rank} repeated {repeated} updates, expected {expected_repeats}")
    return errors


def validate(report):
    import numpy as np
    baseline_mode = report.get("baseline_mode", "ordinary")
    for sample in report["samples"]:
        sample["recovery_errors"] = recovery_errors(sample, report["parameters"])
        sample["recovery_contract_valid"] = not sample["recovery_errors"]
        if sample["status"] != "completed":
            continue
        reference = next(s for s in report["samples"] if s["repeat"] == sample["repeat"]
                         and s["mode"] == baseline_mode and not s["fault"])
        if reference["status"] != "completed":
            sample["validation"] = "missing healthy baseline reference"
            continue
        reference_array = np.load(reference["final"]["0"]["model_path"])
        error = max(float(np.max(np.abs(np.load(v["model_path"]) - reference_array)))
                    for v in sample["final"].values())
        sample["max_parameter_error"] = error
        sample["matches_reference"] = error <= 1e-5
        if report.get("text_input"):
            expected_batches = {(e["rank"], e["step"]): e["digest"]
                                for e in reference["events"] if e["kind"] == "input_batch"}
            actual_batches = [e for e in sample["events"] if e["kind"] == "input_batch"]
            sample["input_matches_reference"] = (
                len(expected_batches) == 2 * report["parameters"]["steps"]
                and {(e["rank"], e["step"]) for e in actual_batches} == set(expected_batches)
                and all(expected_batches.get((e["rank"], e["step"])) == e["digest"] for e in actual_batches))
            sample["matches_reference"] &= sample["input_matches_reference"]
    report["pairs"] = []
    for repeat in range(report["parameters"]["repeats"]):
        samples = {(s["mode"], s["fault"]): s for s in report["samples"] if s["repeat"] == repeat}
        if len(samples) != 4 or not all(s.get("matches_reference") and s.get("recovery_contract_valid") for s in samples.values()):
            report["pairs"].append({"repeat": repeat, "valid": False})
            continue
        ordinary, ours = samples[(baseline_mode, True)], samples[("aligned", True)]
        baseline, healthy = samples[(baseline_mode, False)], samples[("aligned", False)]
        report["pairs"].append({"repeat": repeat, "valid": True,
            "fault_runtime_reduction_pct": 100 * (ordinary["fit_s"] - ours["fit_s"]) / ordinary["fit_s"],
            "healthy_fit_overhead_pct": 100 * (healthy["fit_s"] / baseline["fit_s"] - 1),
            "healthy_loop_overhead_pct": 100 * (healthy["loop_span_s"] / baseline["loop_span_s"] - 1)})



### Plot generation

`plot_step_input_experiment.py` — Creates the PNG/PDF figures from the recorded JSON results.


In [ ]:
%%writefile ray_gcp_harness_visible/plot_step_input_experiment.py
"""Plot saved input-recovery ablation evidence; never launches training."""
import argparse
import json
from pathlib import Path
import statistics


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("report", type=Path)
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args()
    report = json.loads(args.report.expanduser().read_text())
    if report.get("profile") in ("ordinary-ray-peer", "torchft-input-comparison"):
        plot_ordinary(report, args.output)
        return
    if report.get("profile") != "step-input-ablation":
        parser.error("Expected a step-input-ablation report")
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.ticker import MaxNLocator

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    groups = [("independent", False), ("aligned", False),
              ("independent", True), ("aligned", True)]
    labels = ["Independent\nhealthy", "Aligned\nhealthy", "Independent\nnode loss", "Aligned\nnode loss"]
    colors = ["#CC6677", "#4477AA", "#CC6677", "#4477AA"]
    fields = [("max_parameter_error", "Final model error vs healthy aligned reference", "Max absolute parameter error"),
              ("input_mismatches", "Input batches inconsistent with model step", "Mismatched batch requests"),
              ("fit_s", "End-to-end fit time, including recovery", "Seconds (lower is better)")]
    for ax, (field, title, ylabel) in zip(axes, fields):
        for index, (mode, fault) in enumerate(groups):
            samples = [s for s in report["samples"] if s["mode"] == mode and s["fault"] == fault]
            measured = [s for s in samples if s.get("status") == "completed" and field in s]
            values = [s[field] for s in measured]
            if values:
                ax.bar(index, statistics.median(values), color=colors[index], alpha=.5, width=.65)
                for j, sample in enumerate(measured):
                    offset = (j - (len(measured) - 1) / 2) * .08
                    valid = sample.get("matches_reference") is True and sample.get("input_mismatches") == 0
                    ax.scatter(index + offset, sample[field], color=colors[index],
                               marker="o" if valid else "x", zorder=3)
            else:
                ax.text(index, .08, "No measurement", rotation=90, ha="center",
                        transform=ax.get_xaxis_transform(), fontsize=8)
            ax.text(index, 1.01, f"{len(measured)}/{len(samples)} measured",
                    ha="center", transform=ax.get_xaxis_transform(), fontsize=8)
        ax.set_xticks(range(4), labels, fontsize=8)
        ax.set_title(title, fontsize=10, pad=24)
        ax.set_ylabel(ylabel)
        ax.grid(axis="y", alpha=.2)
        ax.set_axisbelow(True)
    axes[0].set_yscale("symlog", linthresh=1e-5)
    axes[0].axhline(1e-5, color="black", linestyle=":", linewidth=1, label="Tolerance 1e-5")
    axes[0].legend(fontsize=8)
    axes[1].yaxis.set_major_locator(MaxNLocator(integer=True))
    fig.suptitle("Contribution ablation: same TorchFT peer recovery, different input recovery")
    p = report["parameters"]
    fig.text(.5, .025,
        f"CPU linear model; two logical worker nodes; rank {p['failed_rank']} killed before update {p['fault_step']}; "
        f"{p['repeats']} repetitions. No durable model checkpoints.\n"
        "Bars: medians; points: individual runs; crosses: incorrect or unvalidated completion. "
        "Missing/failed runs are not plotted as zero.\n"
        "Independent input is a TorchFT ablation, not ordinary Ray full-group checkpoint recovery. "
        "Runtime includes worker startup; no production throughput claim.",
        ha="center", fontsize=8)
    fig.tight_layout(rect=(0, .16, 1, .94))
    output = args.output.expanduser()
    output.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output, dpi=180)
    fig.savefig(output.with_suffix(".pdf"))
    plt.close(fig)
    print(output)


def plot_ordinary(report, output):
    from _support.ordinary_ray_comparison import recovery_errors

    for sample in report["samples"]:
        sample["recovery_contract_valid"] = not recovery_errors(sample, report["parameters"])
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    text_input = report.get("text_input")
    indexed = report.get("profile") == "torchft-input-comparison"
    baseline_mode = "indexed" if indexed else "ordinary"
    baseline_label = ("TorchFT + cursor resume" if text_input else "TorchFT + indexed input") if indexed else "Ordinary Ray\nDDP + checkpoint retry"
    panels = [(False, "fit_s", "Healthy completion time", "Seconds"),
              (True, "fit_s", "Completion time with worker-node loss", "Seconds"),
              (True, "recovery_to_new_progress_s", "Node loss to new model progress", "Seconds"),
              (True, "repeated_updates_per_rank", "Committed updates repeated after rollback", "Mean repeated updates\nper worker")]
    if text_input:
        panels[-1] = (True, "redecoded_chunks", "Input chunks decoded more than once", "Chunks")
        panels.extend([(True, "compressed_bytes_read", "Total compressed input read", "Bytes"),
                       (True, "preprocessing_worker_seconds", "Summed preprocessing task time", "Worker-seconds (not wall time)")])
        for sample in report["samples"]:
            sample.update(sample.get("input_work", {}))
    fig, axes = plt.subplots(3 if text_input else 2, 2, figsize=(12, 11) if text_input else (11, 8))
    for ax, (fault, field, title, ylabel) in zip(axes.flat, panels):
        for index, (mode, color) in enumerate(((baseline_mode, "#CC6677"), ("aligned", "#4477AA"))):
            samples = [s for s in report["samples"] if s["mode"] == mode and s["fault"] == fault]
            validated = [s for s in samples if s.get("matches_reference") is True and s["recovery_contract_valid"] and field in s]
            values = [(sum(s[field].values()) / 2 if field == "repeated_updates_per_rank" else s[field])
                      for s in validated]
            if values:
                ax.bar(index, statistics.median(values), color=color, alpha=.6, width=.55)
                for j, value in enumerate(values):
                    ax.scatter(index + .07 * (j - (len(values) - 1) / 2), value, color=color, zorder=3)
            else:
                ax.text(index, .1, "No validated result", rotation=90, ha="center",
                        transform=ax.get_xaxis_transform())
            ax.text(index, 1.02, f"{len(validated)}/{len(samples)} validated", ha="center",
                    transform=ax.get_xaxis_transform(), fontsize=9)
        ax.set_xlim(-0.6, 1.6)
        ax.set_xticks([0, 1], [baseline_label, "TorchFT + retained\naligned input"])
        ax.set_title(title, pad=25)
        ax.set_ylabel(ylabel)
        ax.set_ylim(bottom=0)
        ax.grid(axis="y", alpha=.2)
        ax.set_axisbelow(True)
    pairs = [p for p in report.get("pairs", []) if p.get("valid") and
             all(s.get("matches_reference") and s["recovery_contract_valid"]
                 for s in report["samples"] if s["repeat"] == p["repeat"])]
    if pairs:
        improvement = statistics.median(p["fault_runtime_reduction_pct"] for p in pairs)
        overhead = statistics.median(p["healthy_fit_overhead_pct"] for p in pairs)
        conclusion = (f"Paired median fault-runtime reduction: {improvement:+.1f}% "
                      f"(negative means slower); healthy fit overhead: {overhead:+.1f}%.")
    else:
        conclusion = "No complete correctness-validated pair; no speedup claim."
    p = report["parameters"]
    delay = p.get("startup_delay_s", 0)
    comparison = "TorchFT input comparison" if indexed else "Ordinary Ray comparison"
    fig.suptitle(f"{comparison} | simulated startup: {delay:g} s per worker invocation")
    fig.text(.5, .025,
        conclusion + "\n" +
        f"{'Real-text' if text_input else 'Synthetic'} CPU MLP width {p['model_width']}; {p['steps']} steps; checkpoint every {p['checkpoint_every']}; "
        f"rank {p['failed_rank']} lost before update {p['fault_step']}; {p['repeats']} repeats.\n" +
        ("Same TorchFT, checkpoints and delay; indexed input reconstructs each batch at the healed model step.\n" if indexed else
         "Same checkpoints and simulated delay in both arms. Ordinary Ray directly resumes input at its checkpoint step.\n") +
        "Bars: medians; points: runs. Only correctness and recovery validated runs contribute timing. Head/storage survive.\n" +
        ("Five GCP VMs, 4 vCPUs each; whole Ray worker service SIGKILL; VM stays up.\n" if p.get("gcp") else "One physical host.\n") +
        (("Local compressed text shards; identical preprocessing; cursor resume versus retained Ray Data delivery."
          if text_input else "Input strategy comparison: cheap deterministic local reconstruction versus retained Ray Data delivery.")
         if indexed else "Combined-system comparison includes TorchFT benefits; the separate input ablation isolates our input contribution."),
        ha="center", fontsize=8)
    fig.tight_layout(rect=(0, .17, 1, .95))
    output = Path(output).expanduser()
    output.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output, dpi=180)
    fig.savefig(output.with_suffix(".pdf"))
    plt.close(fig)
    print(output)


if __name__ == "__main__":
    main()



### Experiment suite runner

`run_gcp_suite.py` — Runs observations in separate processes, records failures and invokes validation and plotting.


In [ ]:
%%writefile ray_gcp_harness_visible/run_gcp_suite.py
"""Sequential matched observations; no cloud instance creation inside the guest."""
import argparse
import hashlib
import json
import os
from pathlib import Path
import signal
import subprocess
import sys
import time
from types import SimpleNamespace


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(data, indent=2))
    tmp.replace(path)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', type=Path, required=True)
    parser.add_argument('--sample', type=Path)
    parser.add_argument('--mode')
    parser.add_argument('--repeat', type=int, default=0)
    parser.add_argument('--fault', action='store_true')
    a = parser.parse_args()
    cfg = json.loads(a.config.read_text())
    if a.sample:
        from _support.ordinary_ray_comparison import run
        run(SimpleNamespace(**cfg, sample=a.sample, mode=a.mode, repeat=a.repeat,
                            fault=a.fault, gcp=True, profile_recovery=False,
                            text_manifest=None, startup_delay_s=0), write_json)
        return
    output = Path(cfg['output'])
    if output.exists():
        raise FileExistsError('Choose a fresh suite name; existing evidence is retained')
    report = {'profile': 'torchft-input-comparison' if cfg['comparison']=='torchft-input' else 'ordinary-ray-peer',
              'baseline_mode': 'indexed' if cfg['comparison']=='torchft-input' else 'ordinary',
              'parameters': dict(cfg, gcp=True, startup_delay_s=0), 'samples': [],
              'scope': 'Five physical GCP VMs; CPU synthetic MLP; head/NFS survive; whole worker Ray service SIGKILL; Fixed-R not enabled',
              'source_commit': json.loads(Path('/etc/ray-gcp.json').read_text())['source_commit'],
              'harness_sha256': {str(p.relative_to(Path(__file__).parent)): hashlib.sha256(p.read_bytes()).hexdigest()
                                 for p in Path(__file__).parent.rglob('*.py')},
              'status': 'running'}
    write_json(output, report)
    baseline = report['baseline_mode']
    for repeat in range(cfg['repeats']):
        for fault in (False, True):
            for mode in ([baseline, 'aligned'] if repeat % 2 == 0 else ['aligned', baseline]):
                folder = output.parent / (output.stem+'-runs') / f"r{repeat}-{mode}-{'fault' if fault else 'healthy'}"
                folder.mkdir(parents=True, exist_ok=False)
                sample = folder/'sample.json'
                cmd = [sys.executable, __file__, '--config', str(a.config), '--sample', str(sample),
                       '--mode', mode, '--repeat', str(repeat)] + (['--fault'] if fault else [])
                print(f'{repeat+1}/{cfg["repeats"]}: {mode} fault={fault}', flush=True)
                timed_out = False
                with (folder/'run.log').open('w') as log:
                    p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
                    try:
                        code = p.wait(timeout=cfg['timeout_s'])
                    except subprocess.TimeoutExpired:
                        timed_out = True
                        os.killpg(p.pid, signal.SIGINT)
                        try:
                            code = p.wait(timeout=20)
                        except subprocess.TimeoutExpired:
                            os.killpg(p.pid, signal.SIGKILL)
                            code = p.wait()
                entry = json.loads(sample.read_text()) if sample.exists() else {}
                if timed_out or code != 0:
                    entry.update(status='timeout' if timed_out else 'failed', exit_code=code)
                entry.update(mode=mode, fault=fault, repeat=repeat, log=str(folder/'run.log'))
                report['samples'].append(entry)
                write_json(output, report)
                from _support.gcp_cluster import GCPCluster
                GCPCluster(folder).shutdown()  # stop even if child never reached finally
                print(entry['status'], flush=True)
    from _support.ordinary_ray_comparison import validate
    validate(report)
    report['status'] = 'finished'
    write_json(output, report)
    subprocess.run([sys.executable, str(Path(__file__).with_name('plot_step_input_experiment.py')),
                    str(output), '--output', str(output.with_suffix('.png'))], check=True)
    valid = all(p.get('valid') for p in report['pairs']) and len(report['pairs']) == cfg['repeats']
    print('ALL PAIRS VALID' if valid else 'INVALID OR FAILED OBSERVATIONS: inspect report/logs', flush=True)
    if not valid:
        sys.exit(2)


if __name__ == '__main__':
    main()


### GCP provisioning, monitoring and deletion

`gcp_launcher.py` — Notebook-side controller for the five VMs, networking, shared storage, suite execution, downloads and optional cleanup.


In [ ]:
%%writefile ray_gcp_harness_visible/gcp_launcher.py
"""Notebook controller. Requires authenticated gcloud + SSH tools on notebook host.
Never modifies the saved image. Cloud resources are recorded before creation.
"""
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
import base64
import hashlib
import json
import os
from pathlib import Path
import re
import shlex
import shutil
import subprocess
import tarfile
import time
import uuid
import zipfile

SOURCE_COMMIT = 'dc4703753c8f3d52e74c389c72449dacc27b41f3'
PYTHON = '/opt/ray-env/ray-dev/bin/python'
ROOT = '/opt/ray-gcp'
SHARED = '/srv/ray-gcp'


def run(cmd, timeout=600, check=True):
    p = subprocess.run([str(c) for c in cmd], capture_output=True, text=True, timeout=timeout)
    if check and p.returncode:
        raise RuntimeError(shlex.join([str(c) for c in cmd])+'\n'+p.stdout[-4000:]+'\n'+p.stderr[-6000:])
    return p


def parallel(function, items, workers=5):
    # Inspect every result, including exceptions, before raising.
    results, errors = {}, []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(function, item): item for item in items}
        for f in as_completed(futures):
            try:
                results[futures[f]] = f.result()
            except Exception as exc:
                errors.append(str(exc))
    if errors:
        raise RuntimeError('\n'.join(errors))
    return results


class GCPExperiment:
    def __init__(self, project='research-488322', zone='us-central1-a', image='ray-built-v1',
                 workdir='ray_gcp_results', state_file=None):
        self.bundle = Path(__file__).parent.resolve()
        self.local = Path(workdir).expanduser().resolve()
        self.local.mkdir(parents=True, exist_ok=True)
        if state_file:
            self.state_path = Path(state_file).resolve()
            self.state = json.loads(self.state_path.read_text())
        else:
            token = datetime.now(timezone.utc).strftime('%m%d-%H%M%S')+'-'+uuid.uuid4().hex[:6]
            ident = 'rayexp-'+token
            self.state_path = self.local/(ident+'-state.json')
            self.state = dict(project=project, zone=zone, image=image, run_id=ident,
                names=[ident+'-head']+[ident+f'-w{i}' for i in range(1,5)],
                network=ident, subnet=ident+'-subnet',
                firewall=[ident+'-internal', ident+'-iap'],
                region=zone.rsplit('-',1)[0], cidr='10.90.0.0/24',
                suites={}, created_at=datetime.now(timezone.utc).isoformat())
            self.save()
        self.project, self.zone = self.state['project'], self.state['zone']
        self.run_id = self.state['run_id']
        self.names = self.state['names']
        self.head = self.names[0]
        self.key = self.local/(self.run_id+'-key')
        print('Experiment:', self.run_id, '\nState file:', self.state_path)

    def save(self):
        temp = self.state_path.with_suffix('.tmp')
        temp.write_text(json.dumps(self.state, indent=2))
        temp.replace(self.state_path)

    def gc(self, *args, timeout=600, check=True):
        return run(['gcloud', *args, '--project='+self.project, '--quiet'], timeout, check)

    def ssh(self, name, command, timeout=600, check=True):
        return self.gc('compute', 'ssh', name, '--zone='+self.zone,
            '--tunnel-through-iap', '--command='+command,
            '--ssh-flag=-oConnectTimeout=15', timeout=timeout, check=check)

    def scp(self, source, destination, timeout=600):
        return self.gc('compute', 'scp', str(source), str(destination),
                       '--zone='+self.zone, '--tunnel-through-iap', timeout=timeout)

    def _check_owner(self, name):
        if name not in self.names:
            raise ValueError('Unrecorded VM')
        p = self.gc('compute', 'instances', 'describe', name, '--zone='+self.zone,
                    '--format=json', check=False)
        if p.returncode:
            if 'not found' in p.stderr.lower() or 'was not found' in p.stderr.lower():
                return None
            raise RuntimeError(p.stderr)
        instance = json.loads(p.stdout)
        if instance.get('labels', {}).get('ray-experiment') != self.run_id:
            raise RuntimeError('Refusing operation on foreign VM '+name)
        return instance

    def provision(self):
        if self.state.get('provision_started'):
            raise RuntimeError('Provisioning was already attempted. Use setup() if VMs exist, or cleanup before a new experiment.')
        for executable in ('gcloud', 'ssh', 'ssh-keygen'):
            if not shutil.which(executable):
                raise RuntimeError('Install '+executable+' on the notebook host first')
        image = json.loads(self.gc('compute', 'images', 'describe', self.state['image'], '--format=json').stdout)
        if image.get('status') != 'READY':
            raise RuntimeError('Image is not READY')
        self.state['image_id'] = image['id']
        self.state['image_self_link'] = image['selfLink']
        self.state['provision_started'] = True
        self.save()
        print('Creating isolated network and five e2-standard-4 VMs ...', flush=True)
        self.gc('compute', 'networks', 'create', self.state['network'], '--subnet-mode=custom',
                '--description='+self.run_id)
        self.gc('compute', 'networks', 'subnets', 'create', self.state['subnet'],
                '--network='+self.state['network'], '--region='+self.state['region'],
                '--range='+self.state['cidr'])
        self.gc('compute', 'firewall-rules', 'create', self.state['firewall'][0],
                '--network='+self.state['network'], '--allow=tcp,udp,icmp',
                '--source-tags='+self.run_id, '--target-tags='+self.run_id)
        self.gc('compute', 'firewall-rules', 'create', self.state['firewall'][1],
                '--network='+self.state['network'], '--allow=tcp:22',
                '--source-ranges=35.235.240.0/20', '--target-tags='+self.run_id)
        self.gc('compute', 'instances', 'create', *self.names, '--zone='+self.zone,
                '--machine-type=e2-standard-4', '--provisioning-model=STANDARD',
                '--image='+self.state['image'], '--image-project='+self.project,
                '--boot-disk-type=pd-balanced', '--boot-disk-size='+str(image['diskSizeGb'])+'GB',
                '--subnet='+self.state['subnet'], '--tags='+self.run_id,
                '--labels=ray-experiment='+self.run_id,
                '--metadata=enable-oslogin=FALSE', '--no-service-account', '--no-scopes', timeout=900)
        nodes = []
        for name in self.names:
            instance = self._check_owner(name)
            nodes.append(dict(name=name, ip=instance['networkInterfaces'][0]['networkIP'], head=name==self.head))
        self.state['nodes'] = nodes
        self.save()
        print('VMs created. Preparing shared storage and environment checks ...', flush=True)
        self.setup()

    def _wait_ssh(self, name):
        deadline = time.monotonic()+240
        last = ''
        while time.monotonic()<deadline:
            try:
                p = self.ssh(name, 'true', timeout=60, check=False)
                if p.returncode == 0:
                    return
                last = p.stderr
            except subprocess.TimeoutExpired:
                last = 'SSH connection attempt timed out'
            time.sleep(5)
        raise RuntimeError(name+' SSH unavailable. Check IAP Tunnel User / Compute permissions and OS Login policy.\n'+last)

    def setup(self):
        if 'nodes' not in self.state:
            # Recover descriptors after a partial batch create.
            nodes = []
            for name in self.names:
                obj = self._check_owner(name)
                if not obj or obj['status'] != 'RUNNING':
                    raise RuntimeError('All five recorded VMs must be running')
                nodes.append(dict(name=name, ip=obj['networkInterfaces'][0]['networkIP'], head=name==self.head))
            self.state['nodes'] = nodes
            self.save()
        for name in self.names:
            self._check_owner(name)
        # First connection sequentially initializes gcloud's SSH key once.
        self._wait_ssh(self.head)
        parallel(self._wait_ssh, self.names[1:])
        if not self.key.exists():
            run(['ssh-keygen', '-t', 'ed25519', '-N', '', '-f', self.key, '-C', self.run_id])
        archive = self.local/(self.run_id+'-harness.zip')
        with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
            for p in self.bundle.rglob('*.py'):
                z.write(p, p.relative_to(self.bundle))
        nodes = self.state['nodes']
        head_ip = nodes[0]['ip']
        public_key = self.key.with_suffix(self.key.suffix+'.pub').read_text().strip()
        # Dedicated, identical UID permits NFS access across hosts.
        base_setup = '''set -e
sudo apt-get update -qq
sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq nfs-common unzip
if ! id raybench >/dev/null 2>&1; then
  sudo useradd --uid 25001 --create-home --home-dir /var/lib/raybench --shell /bin/bash raybench
fi
test "$(id -u raybench)" = 25001
sudo mkdir -p /opt/ray-gcp /srv/ray-gcp
mountpoint -q /srv/ray-gcp || sudo chown raybench:raybench /srv/ray-gcp
'''
        def setup_one(name):
            node = next(n for n in nodes if n['name']==name)
            config = dict(node, nodes=nodes, head_ip=head_ip, run_id=self.run_id, source_commit=SOURCE_COMMIT)
            config_path = self.local/(name+'-config.json')
            config_path.write_text(json.dumps(config))
            self.ssh(name, base_setup, timeout=600)
            self.scp(archive, name+':/tmp/'+self.run_id+'.zip')
            self.scp(config_path, name+':/tmp/'+self.run_id+'.json')
            self.ssh(name, 'set -e\n'+
                f'sudo unzip -oq /tmp/{self.run_id}.zip -d {ROOT}\n'+
                f'sudo install -o root -g root -m 644 /tmp/{self.run_id}.json /etc/ray-gcp.json\n'+
                f'sudo chown -R root:root {ROOT}\n'+
                f'sudo chmod -R a+rX,go-w {ROOT}\n'+
                f'rm /tmp/{self.run_id}.zip /tmp/{self.run_id}.json\n')
            if not node['head']:
                # Key can only execute the small root-owned node controller,
                # and only from the experiment's head internal IP.
                forced = f'from="{head_ip}",restrict,command="{PYTHON} {ROOT}/node_control.py" {public_key}\n'
                encoded = base64.b64encode(forced.encode()).decode()
                self.ssh(name, 'set -e\nsudo mkdir -p /root/.ssh\nsudo chmod 700 /root/.ssh\n'+
                    f"printf %s {shlex.quote(encoded)} | base64 -d | sudo tee -a /root/.ssh/authorized_keys >/dev/null\n"+
                    'sudo chmod 600 /root/.ssh/authorized_keys\n')
            return self.ssh(name, 'cat /etc/ssh/ssh_host_ed25519_key.pub').stdout.strip()
        hostkeys = parallel(setup_one, self.names)
        exports = SHARED+' '+ ' '.join(n['ip']+'(rw,sync,no_subtree_check,root_squash,fsid=0)' for n in nodes[1:])+'\n'
        encoded = base64.b64encode(exports.encode()).decode()
        sudoers = f'raybench ALL=(root) NOPASSWD: {PYTHON} {ROOT}/node_control.py\n'
        encoded_sudo = base64.b64encode(sudoers.encode()).decode()
        self.ssh(self.head, 'set -e\nsudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq nfs-kernel-server\nsudo install -d -m 755 /etc/exports.d\n'+
            f'printf %s {shlex.quote(encoded)} | base64 -d | sudo tee /etc/exports.d/ray-gcp.exports >/dev/null\n'+
            'sudo exportfs -ra\nsudo systemctl enable --now nfs-server\n'+
            f'printf %s {shlex.quote(encoded_sudo)} | base64 -d | sudo tee /etc/sudoers.d/ray-gcp >/dev/null\n'+
            'sudo chmod 440 /etc/sudoers.d/ray-gcp\nsudo visudo -cf /etc/sudoers.d/ray-gcp\n'+
            'sudo install -d -o raybench -g raybench -m 700 /var/lib/raybench/.ssh\n')
        self.scp(self.key, self.head+':/tmp/'+self.run_id+'-key')
        known = self.local/(self.run_id+'-known-hosts')
        known.write_text(''.join(n['ip']+' '+hostkeys[n['name']]+'\n' for n in nodes[1:]))
        self.scp(known, self.head+':/tmp/'+self.run_id+'-known')
        self.ssh(self.head, 'set -e\n'+
            f'sudo install -o raybench -g raybench -m 600 /tmp/{self.run_id}-key /var/lib/raybench/.ssh/id_ed25519\n'+
            f'sudo install -o raybench -g raybench -m 600 /tmp/{self.run_id}-known /var/lib/raybench/.ssh/known_hosts\n'+
            f'rm /tmp/{self.run_id}-key /tmp/{self.run_id}-known\n')
        def mount(name):
            self.ssh(name, 'set -e\n'+
                f'mountpoint -q {SHARED} || sudo mount -t nfs4 -o vers=4.2,hard {head_ip}:/ {SHARED}\n'+
                f'sudo -u raybench touch {SHARED}/write-check-{name}\n')
        parallel(mount, self.names[1:])
        # Direct SSH must work before any long-running job or billable sweep.
        check_code = ('from _support.gcp_cluster import command; import json; '+
                      'c=json.load(open("/etc/ray-gcp.json")); '+
                      'print(json.dumps([command(n,"identity") for n in c["nodes"]]))')
        result = self.ssh(self.head, f'cd {ROOT} && sudo -u raybench {PYTHON} -c '+shlex.quote(check_code), timeout=600)
        identities = json.loads(result.stdout)
        if any(i['git_commit'] != SOURCE_COMMIT for i in identities):
            raise RuntimeError('Image checkout differs from the inspected benchmark commit; inspect identity report')
        self.state['identities'] = identities
        self.state['ready'] = True
        self.save()
        print('Five VMs ready. Ray/TorchFT imports, direct SSH and shared storage passed.', flush=True)

    def run_suite(self, name, comparison='ordinary-ray', repeats=3, steps=300,
                  batch_size=256, model_width=2048, checkpoint_every=120,
                  fault_step=239, failed_rank=0, timeout_s=1800):
        if not self.state.get('ready'):
            raise RuntimeError('Run provision/setup first')
        if not re.fullmatch(r'[a-z0-9][a-z0-9-]{0,45}', name):
            raise ValueError('Use a short lowercase suite name')
        if comparison not in ('ordinary-ray', 'torchft-input'):
            raise ValueError('Unsupported comparison')
        if min(repeats, steps, batch_size, model_width, checkpoint_every, timeout_s)<=0:
            raise ValueError('Positive parameters required')
        if not 0<fault_step<steps or fault_step%checkpoint_every==0 or failed_rank not in (0,1):
            raise ValueError('Failure must be strictly between checkpoints and steps')
        if name in self.state['suites']:
            raise RuntimeError('Suite already recorded; use wait_suite/download or a new name')
        remote_dir = SHARED+'/'+self.run_id+'/'+name
        unit = 'ray-exp-driver-'+name
        config = dict(comparison=comparison, repeats=repeats, steps=steps, batch_size=batch_size,
                      model_width=model_width, checkpoint_every=checkpoint_every,
                      fault_step=fault_step, failed_rank=failed_rank, timeout_s=timeout_s,
                      output=remote_dir+'/report.json')
        # Refuse concurrent suites; all observations reuse these same five VMs.
        active = self.ssh(self.head, "systemctl list-units 'ray-exp-driver-*' --state=running --no-legend").stdout
        if active.strip():
            raise RuntimeError('Another suite is active: '+active)
        self.state['suites'][name] = dict(config=config, unit=unit, remote=remote_dir, state='launching')
        self.save()
        local_config = self.local/(name+'-'+self.run_id+'.json')
        local_config.write_text(json.dumps(config))
        self.scp(local_config, self.head+':/tmp/'+self.run_id+'-suite.json')
        # RuntimeMaxSec bounds a detached suite if the notebook disconnects.
        maximum = repeats*4*(timeout_s+300)+600
        script = (f'{PYTHON} -u {ROOT}/run_gcp_suite.py --config {remote_dir}/config.json '
                  f'> {remote_dir}/suite.log 2>&1\n'
                  'status=$?\n'
                  f'printf "%s\\n" "$status" > {remote_dir}/exit-code.tmp\n'
                  f'mv {remote_dir}/exit-code.tmp {remote_dir}/exit-code\n'
                  'exit "$status"\n')
        script_path = self.local/(name+'-'+self.run_id+'.sh')
        script_path.write_text(script)
        self.scp(script_path, self.head+':/tmp/'+self.run_id+'-suite.sh')
        self.ssh(self.head, 'set -e\n'+
            f'sudo -u raybench mkdir -p {remote_dir}\n'+
            f'sudo install -o raybench -g raybench -m 644 /tmp/{self.run_id}-suite.json {remote_dir}/config.json\n'+
            f'sudo install -o raybench -g raybench -m 644 /tmp/{self.run_id}-suite.sh {remote_dir}/run.sh\n'+
            f'rm /tmp/{self.run_id}-suite.json /tmp/{self.run_id}-suite.sh\n'+
            f'sudo systemd-run --quiet --collect --unit={unit} --uid=raybench --gid=raybench '+
            f'--property=KillMode=control-group --property=RuntimeMaxSec={maximum} '+
            '--setenv=PATH=/opt/ray-env/ray-dev/bin:/usr/bin:/bin '+
            '--setenv=RAY_TRAIN_V2_ENABLED=1 --setenv=RAY_USAGE_STATS_ENABLED=0 '+
            '--setenv=MPLBACKEND=Agg --setenv=OMP_NUM_THREADS=1 '+
            '--setenv=MKL_NUM_THREADS=1 --setenv=OPENBLAS_NUM_THREADS=1 '+
            f'/bin/bash {remote_dir}/run.sh')
        self.state['suites'][name]['state']='running'
        self.save()
        return self.wait_suite(name)

    def wait_suite(self, name):
        suite=self.state['suites'][name]
        last=''
        while True:
            code = (f'if test -f {suite["remote"]}/exit-code; then cat {suite["remote"]}/exit-code; '+
                    f'else systemctl is-active {suite["unit"]} || true; fi')
            status=self.ssh(self.head, code).stdout.strip()
            tail=self.ssh(self.head, f'tail -n 8 {suite["remote"]}/suite.log 2>/dev/null || true').stdout
            if tail!=last:
                print(tail, flush=True)
                last=tail
            if status.isdigit():
                suite['exit_code']=int(status)
                suite['state']='finished' if status=='0' else 'failed'
                self.save()
                destination=self.download(name)
                if status!='0':
                    raise RuntimeError(f'Suite failed/invalid (exit {status}); inspect {destination}. Full sweep blocked.')
                return destination
            if status not in ('active', 'activating'):
                suite['state']='interrupted'
                self.save()
                self.download(name)
                raise RuntimeError('Driver service ended without a completion marker; inspect logs. Use stop_services() before retry.')
            time.sleep(15)

    def download(self, name):
        suite=self.state['suites'][name]
        destination=self.local/self.run_id/name
        destination.mkdir(parents=True, exist_ok=True)
        # Shared results belong to raybench, not the gcloud SSH login user.
        # Create a private archive as that owner, then grant only the login user
        # read access for SCP. Never relax permissions on the results directory.
        remote_archive=SHARED+'/'+self.run_id+'-'+name+'-'+uuid.uuid4().hex+'.tar.gz'
        quoted_archive=shlex.quote(remote_archive)
        tar_command=shlex.join(['tar', '-czf', remote_archive, '-C', suite['remote'], '.'])
        archive=destination/'evidence.tar.gz'
        try:
            self.ssh(self.head, 'set -e\n' +
                'sudo -n -u raybench sh -c ' + shlex.quote('umask 077; exec '+tar_command) + '\n' +
                'sudo -n chown "$(id -u):$(id -g)" ' + quoted_archive, timeout=600)
            self.scp(self.head+':'+remote_archive, archive, timeout=900)
            with tarfile.open(archive) as tar:
                # Archive comes from our own results dir; reject links/traversal.
                for member in tar.getmembers():
                    target=(destination/member.name).resolve()
                    if not target.is_relative_to(destination.resolve()) or member.issym() or member.islnk():
                        raise ValueError('Unexpected archive member')
                tar.extractall(destination)
        finally:
            # Parent-directory owner can remove the login-owned temporary file.
            # A failed transfer can be retried without rerunning the experiment.
            try:
                cleanup=self.ssh(self.head,
                    'sudo -n -u raybench rm -f -- '+quoted_archive, check=False)
                if cleanup.returncode:
                    print('Temporary archive cleanup failed:', remote_archive)
            except Exception as exc:
                print('Temporary archive cleanup failed:', remote_archive, str(exc))
        suite['downloaded']=str(destination)
        suite['archive_sha256']=hashlib.sha256(archive.read_bytes()).hexdigest()
        self.save()
        return destination

    def stop_services(self):
        """Stop detached drivers and Ray services, retaining VMs/results."""
        for name in self.names:
            obj=self._check_owner(name)
            if not obj or obj['status']!='RUNNING':
                continue
            if name==self.head:
                for suite in self.state['suites'].values():
                    self.ssh(name, f'sudo systemctl stop {suite["unit"]} || true')
            payload=shlex.quote(json.dumps(dict(run_id=self.run_id, action='stop')))
            self.ssh(name, f'printf %s {payload} | sudo {PYTHON} {ROOT}/node_control.py')
        print('Experiment services stopped. VMs still incur charges.')

    def cleanup(self, confirm=None, delete_network=True, allow_uncollected=False):
        """Delete ONLY manifest-named, label-matching VMs and their boot disks."""
        if confirm!=self.run_id:
            raise ValueError('Pass confirm=exp.run_id to explicitly delete this experiment')
        missing=[n for n,s in self.state['suites'].items() if not s.get('downloaded')]
        if missing and not allow_uncollected:
            raise RuntimeError('Download these suites first, or pass allow_uncollected=True to discard their remote evidence: '+str(missing))
        existing=[n for n in self.names if self._check_owner(n) is not None]
        if existing:
            self.gc('compute', 'instances', 'delete', *existing, '--zone='+self.zone,
                    '--delete-disks=boot', timeout=900)
        if delete_network:
            # Names are unique and stored before creation; absence is harmless.
            for firewall in self.state['firewall']:
                p=self.gc('compute', 'firewall-rules', 'describe', firewall, '--format=json', check=False)
                if p.returncode==0:
                    obj=json.loads(p.stdout)
                    if obj['network'].rsplit('/',1)[-1]!=self.state['network']:
                        raise RuntimeError('Foreign firewall rule')
                    self.gc('compute','firewall-rules','delete',firewall)
                elif 'not found' not in p.stderr.lower():
                    raise RuntimeError(p.stderr)
            p=self.gc('compute','networks','subnets','describe',self.state['subnet'],
                      '--region='+self.state['region'],'--format=json',check=False)
            if p.returncode==0:
                if json.loads(p.stdout)['network'].rsplit('/',1)[-1]!=self.state['network']:
                    raise RuntimeError('Foreign subnet')
                self.gc('compute','networks','subnets','delete',self.state['subnet'],'--region='+self.state['region'])
            elif 'not found' not in p.stderr.lower():
                raise RuntimeError(p.stderr)
            p=self.gc('compute','networks','describe',self.state['network'],'--format=json',check=False)
            if p.returncode==0:
                if json.loads(p.stdout).get('description')!=self.run_id:
                    raise RuntimeError('Foreign network')
                self.gc('compute','networks','delete',self.state['network'])
            elif 'not found' not in p.stderr.lower():
                raise RuntimeError(p.stderr)
        self.state['deleted_at']=datetime.now(timezone.utc).isoformat()
        self.save()
        print('Experiment VMs, boot disks and requested network resources deleted. Image retained:',self.state['image'])


### Detailed experiment notes

These notes are also written beside the helper scripts.


In [ ]:
%%writefile ray_gcp_harness_visible/README.md
# GCP Ray recovery experiments

Project: research-488322. Image: ray-built-v1. Zone: us-central1-a.

Run the self-contained notebook on a local Linux/macOS Jupyter host (or a suitable
cloud notebook) with Python >=3.10, Google Cloud CLI, OpenSSH and an authenticated
Google account. The controller uses Python's standard library; IPython displays
plots. The notebook host does not need Ray, Torch or a GPU. This controller must
remain outside the five disposable experiment VMs.

The account needs Compute instance/image/network/firewall management, metadata/SSH
access and IAP-secured Tunnel User access. Enable compute.googleapis.com and
iap.googleapis.com. The notebook uses a separate VPC, IAP-only SSH ingress, and
internal communication between tagged experiment VMs. External IPs provide package
download access. Guest VMs have no Google service account. The code requests
metadata-based SSH (`enable-oslogin=FALSE`); an organization enforcing OS Login or
blocking external IPs/root key login needs a different connection setup. It fails
at preflight rather than silently changing organization policy.

## What runs

Five e2-standard-4 Standard VMs (4 vCPUs, 16 GB each). One survives as head,
recorder and NFSv4 server. Two training ranks reserve three CPUs on distinct worker
VMs; two additional workers provide spare capacity. A workload consists of a
seeded synthetic CPU MLP. It is not a real dataset or GPU/LLM benchmark.

The launcher creates a fresh Ray cluster for every observation. A deterministic
pre-update gate pauses the selected rank. From the head, the driver sends a direct
SSH request to a forced command on that worker. `systemctl kill --kill-whom=all
--signal=SIGKILL ray-exp-node.service` kills the complete service cgroup, including
actors and object store. Restart is disabled. The affected worker service stays
down for the observation. This models abrupt loss of the Ray runtime on a separate
VM; the VM itself, its disk and the head's NFS server remain up. This is not a
GCP VM power-off, Spot eviction, storage-loss or head-failure test.

The temporary inter-VM key is usable only from the head IP for the root-owned node
controller. Host keys are collected through authenticated gcloud SSH, and direct
SSH verifies them. All keys/configuration are confined to disposable VMs and the
local experiment directory. The image and existing repository are not modified.
The harness is uploaded separately under /opt/ray-gcp. A dedicated raybench UID
25001 is used consistently across the five NFS clients; a conflicting UID fails
setup.

## Comparisons

1. Ordinary Ray DDP/full-group checkpoint retry vs TorchFT + aligned Ray Data input,
   at checkpoint intervals 30, 60 and 120.
2. The same TorchFT peer recovery with indexed input vs retained aligned input,
   at checkpoint interval 120.

Default full sweep: 300 steps, fault before update 239, batch 256 per rank, width
2048, three seeds/repetitions, zero simulated startup delay. Each comparison has
healthy/failure observations for both methods (12 observations per suite; 48 for
four suites). The smoke tests use width 512, batch 32, 60 steps, checkpoint 30,
fault step 49, one repeat (4 observations per comparison). Sweeps run sequentially
on the same five VMs.

This is the same fork in both arms. TorchFT supplies model/optimizer peer state
recovery. The first comparison measures the combined integration; only the second
controls for TorchFT and isolates the input strategy. Fixed-R is not enabled by
this harness. No claim that retention must improve performance is built into the
analysis.

## Evidence

Per-suite report.json, report.png/PDF, suite.log, per-observation logs, cluster
mapping, final model arrays and checkpoints are retained. Plots show healthy
runtime, failure runtime, recovery-to-new-progress and repeated committed updates.
Bars are medians, points are repetitions, and validated/total counts expose failed
runs. Final models must agree with the same-seed healthy baseline within 1e-5;
restart/checkpoint/rollback events must satisfy the original recovery contract.
Failed correctness blocks the full sweep; do not relax tolerance to manufacture
speedups. E2 hosts can use different CPU platforms; investigate numerical failures.

All event timestamps are assigned on recorder arrival at the head. Recovery starts
at the head driver's kill request and includes SSH dispatch/acknowledgement;
cluster setup is outside fit time. Training startup and recovery are included.
NFS checkpoint cost and head contention are shared by both arms. Fine-grained
cross-host profiling is deliberately not inferred from unsynchronized clocks.
Identity records include source commit/hashes, package versions and _raylet hash.
The image must contain fork commit dc4703753c8f3d52e74c389c72449dacc27b41f3;
the local RocksDB parallel-build edit is compatible with this requirement.

The benchmark core and plotter are adapted from that commit's
`gossip_benchmarks/_support/ordinary_ray_comparison.py` and
`gossip_benchmarks/plot_step_input_experiment.py`. Changes are confined to physical
cluster transport, an explicit head-IP Lighthouse bind, and the plot's cloud scope
label. The training/recovery algorithm and validation are retained.

## Persistence and cleanup

Driver suites run in detached systemd services. A notebook interruption does not
stop them. Use `exp.wait_suite(name)` to reconnect, `exp.download(name)` to fetch
partial evidence or `exp.stop_services()` to stop work. Service time limits bound
an individual suite but do not delete or shut down billable VMs.

Save the printed state file. After restarting the kernel, rerun the visible source
cells and construct `GCPExperiment(state_file=..., workdir=...)` with the same
local results directory; do not provision a second set of VMs accidentally.

Downloads preserve remote absolute paths in JSON as provenance. Use the included
plotter on the guest to regenerate plots while model files are available there;
local plots are already downloaded. Evidence archives include model arrays and
checkpoints and can be several GB. Ensure sufficient disk space on the notebook
host. Archives are copied before cleanup.

`exp.cleanup(confirm=exp.run_id)` deletes only recorded, matching-label instances
and their boot disks, then the recorded experiment firewall/subnet/VPC. It does
not delete the custom image or any unrelated VM. It refuses deletion when a suite
has not been downloaded; `allow_uncollected=True` explicitly discards uncollected
remote evidence. Use this override to clean up after failed provisioning if
needed. VMs, disks and external IPs incur charges until cleanup; the saved custom
image continues to incur storage charges.

## Validation performed here

Python compilation, visible notebook code checks, mocked resource creation and
cleanup ownership checks, and plot/validation checks. No GCP VM was launched and
no distributed benchmark was executed by the authoring assistant. The smoke run
is the first live integration test, not a formality.


### Import the controller

The scripts above are now available as ordinary Python files.


In [ ]:
sys.path.insert(0, str(BUNDLE_DIR))
from gcp_launcher import GCPExperiment
print("Controller imported from:", BUNDLE_DIR / "gcp_launcher.py")


## 3. Configure and create the experiment

Leave `STATE_FILE = None` for a **new** experiment. After a kernel restart, set it to the printed `*-state.json` file to reconnect to the existing experiment. Do not create a second cluster to resume a first one.

The state file records exact VM names, cloud configuration, suite status and downloaded results. Keep it until cleanup completes.

**Resuming after setup failure:** set `STATE_FILE` below to the existing state JSON before running the configuration cell. The following setup cell resumes the recorded VMs automatically. Restart the kernel after pulling this notebook so the updated controller is imported.


In [ ]:
PROJECT = "research-488322"
ZONE = "us-central1-a"
IMAGE = "ray-built-v1"
WORKDIR = "ray_gcp_results"
STATE_FILE = None  # To resume: "ray_gcp_results/rayexp-...-state.json"

exp = GCPExperiment(project=PROJECT, zone=ZONE, image=IMAGE,
                    workdir=WORKDIR, state_file=STATE_FILE)
print("Planned instances:", *exp.names, sep="\n  ")


**The next cell starts billable resources.** It creates the five VMs, installs NFS packages, sets up shared checkpoint storage on the head, and checks imports and the pinned source revision. All nodes use the same image and software paths.

If setup fails after creation, inspect the error. `exp.setup()` retries setup on the same VMs; do not rerun provisioning. The cleanup cell also works after partial provisioning with the explicit discard option if necessary.


In [ ]:
if exp.state.get("ready"):
    print("Existing experiment is ready; provisioning/setup skipped.")
elif exp.state.get("provision_started"):
    print("Resuming setup on the existing five VMs.")
    exp.setup()
else:
    exp.provision()


## 4. Smoke tests — run before the full sweep

Each smoke comparison has four observations: both methods, healthy and failed. Both use 60 steps, checkpoint interval 30, failure before update 49, batch 32 and width 512. Each observation gets a fresh Ray cluster.

A passing suite requires model agreement, correct checkpoint restoration and rollback counts, the expected worker restarts, and training ranks on different worker VMs. The driver runs on the head, so interrupting this notebook only stops monitoring; use the interruption controls below to stop the remote work.


In [ ]:
def run_or_resume(name, **kwargs):
    previous = exp.state["suites"].get(name)
    if previous is None:
        return exp.run_suite(name, **kwargs)
    if previous.get("state") == "finished" and previous.get("exit_code") == 0:
        path = Path(previous.get("downloaded", ""))
        return path if (path / "report.json").exists() else exp.download(name)
    if previous.get("state") in ("running", "launching"):
        return exp.wait_suite(name)
    raise RuntimeError(f"{name} previously failed/interrupted. Inspect evidence before choosing a fresh suite name.")

smoke_options = dict(repeats=1, steps=60, batch_size=32, model_width=512,
                     checkpoint_every=30, fault_step=49, timeout_s=1200)
smoke_ordinary = run_or_resume("smoke-ordinary", comparison="ordinary-ray", **smoke_options)
smoke_input = run_or_resume("smoke-input", comparison="torchft-input", **smoke_options)
print("Both smoke suites passed.")


In [ ]:
from IPython.display import display, Image, Markdown

def show_suite(name):
    folder = Path(exp.state["suites"][name]["downloaded"])
    report = json.loads((folder / "report.json").read_text())
    samples = report["samples"]
    valid = sum(s.get("matches_reference") is True and s.get("recovery_contract_valid") is True
                for s in samples)
    display(Markdown(f"### {name}: {valid}/{len(samples)} validated observations"))
    if (folder / "report.png").exists():
        display(Image(filename=str(folder / "report.png")))
    print("JSON, PNG/PDF, checkpoints and logs:", folder)
    return report

show_suite("smoke-ordinary")
show_suite("smoke-input");


## 5. Full sweep — enable after both smoke tests pass

Set `RUN_FULL_SUITE = True`. Four suites run sequentially on the **same five VMs**. Each suite uses three repetitions (12 observations), for 48 observations total. The runner alternates method order by repetition. This can take hours; completion time depends on recovery delays and your VMs.

All runs use 300 steps, failure before update 239, batch 256, width 2048 and no artificial startup delay. The full sweep stops if a suite has invalid/failed observations. It does not retry failures until they look successful.


In [ ]:
RUN_FULL_SUITE = False

if RUN_FULL_SUITE:
    for smoke in ("smoke-ordinary", "smoke-input"):
        state = exp.state["suites"].get(smoke, {})
        if state.get("state") != "finished" or state.get("exit_code") != 0:
            raise RuntimeError("Both smoke suites must pass before the full sweep.")
    for interval in (30, 60, 120):
        run_or_resume(f"ordinary-cp{interval}", comparison="ordinary-ray",
                      repeats=3, steps=300, batch_size=256, model_width=2048,
                      checkpoint_every=interval, fault_step=239, timeout_s=1800)
    run_or_resume("input-cp120", comparison="torchft-input",
                  repeats=3, steps=300, batch_size=256, model_width=2048,
                  checkpoint_every=120, fault_step=239, timeout_s=1800)
else:
    print("Full sweep skipped. Set RUN_FULL_SUITE = True when ready.")


## 6. Display results

Every figure includes healthy runtime, runtime under worker service loss, recovery to new progress, and repeated committed updates. Bars show medians with individual-run points and validated/total counts. Only complete valid pairs support the paired percentage changes.

A **positive failure-runtime reduction** means faster recovery runs; a **positive healthy overhead** means slower healthy runs. Retain negative results. The ordinary-Ray comparison is a combined-system comparison, not evidence that input retention alone caused a speedup.


In [ ]:
import statistics

rows = ["| Suite | Valid pairs | Median failure-runtime reduction | Median healthy overhead |",
        "|---|---:|---:|---:|"]
for name in ("ordinary-cp30", "ordinary-cp60", "ordinary-cp120", "input-cp120"):
    state = exp.state["suites"].get(name, {})
    folder = Path(state.get("downloaded", "/nonexistent"))
    if not (folder / "report.json").exists():
        continue
    report = show_suite(name)
    pairs = [p for p in report.get("pairs", []) if p.get("valid")]
    if pairs:
        reduction = statistics.median(p["fault_runtime_reduction_pct"] for p in pairs)
        overhead = statistics.median(p["healthy_fit_overhead_pct"] for p in pairs)
        rows.append(f"| {name} | {len(pairs)}/{report['parameters']['repeats']} | {reduction:+.1f}% | {overhead:+.1f}% |")
    else:
        rows.append(f"| {name} | 0 | No valid comparison | No valid comparison |")
display(Markdown("\n".join(rows)))


## 7. Reconnect, stop, or collect partial results

If the notebook disconnects, the current suite continues on the head with a bounded service lifetime. The VMs remain running afterward. Reload this notebook's source-writing/configuration cells using the saved `STATE_FILE`, then use the appropriate command below.

Do not call `stop_services()` while you want the suite to continue. It stops only this experiment's jobs; it does not delete VMs.

If collection failed after a suite ended, call `exp.download(name)` to retry collection, then `exp.wait_suite(name)` to inspect its recorded exit status. This does not rerun training.


In [ ]:
# Uncomment only the operation you need:
# exp.wait_suite("ordinary-cp120")   # Continue monitoring an already launched suite.
# exp.stop_services()               # Stop remote work but retain VMs and results.
# exp.download("ordinary-cp120")     # Download completed or partial evidence.

print("State file:", exp.state_path)
print("Recorded suites:", {n: s.get("state") for n, s in exp.state["suites"].items()})


## 8. Optional deletion — all instances created by this notebook

Set `DELETE_INSTANCES = True` to delete this experiment's **five VMs and their boot disks**, plus its firewall rules, subnet and VPC. The launcher checks the recorded names and experiment labels. It never deletes `ray-built-v1`, your project, or unrelated instances.

Collect results first. If setup failed or you deliberately want to discard remote evidence, also set `DISCARD_UNCOLLECTED = True`. This override permanently discards uncollected results on those VM disks. The local downloads remain.


In [ ]:
DELETE_INSTANCES = False
DISCARD_UNCOLLECTED = False

if DELETE_INSTANCES:
    exp.cleanup(confirm=exp.run_id, allow_uncollected=DISCARD_UNCOLLECTED)
else:
    print("Cleanup skipped. Experiment VMs remain billable.")
    print("Saved image is retained either way:", exp.state["image"])


## Scope and provenance

The bundled README documents the failure model, timing semantics, NFS storage, permissions and recovery checks. Read it with:

```python
print((BUNDLE_DIR / "README.md").read_text())
```

The benchmark core comes from your fork at `dc4703753c8f3d52e74c389c72449dacc27b41f3`. Its model/recovery checks are retained. GCP transport and an explicit private head-IP Lighthouse binding are added. Source, package and native-extension identities are recorded on the VMs.

Reference documentation: [GCP custom images](https://cloud.google.com/compute/docs/instances/create-vm-from-custom-image), [IAP TCP forwarding](https://cloud.google.com/iap/docs/using-tcp-forwarding), [instance deletion](https://cloud.google.com/sdk/gcloud/reference/compute/instances/delete).
